# validation-no-grad — faded example 2: Accumulate correct count under no_grad

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `validation-no-grad`. Running the beacon reports progress on the `PyTorch: no_grad validation` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: no_grad validation` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`validation-no-grad`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "validation-no-grad"
DD_SUBTOPIC = "PyTorch: no_grad validation"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

An eval loop under `no_grad` adds the per-batch number of correct predictions into a running integer. The prediction is `logits.argmax(dim=1)`, and matches are counted by summing the boolean equality with the labels.

## Faded exercise 2

### Count correct predictions per batch

Implement `count_correct(model, batches)` returning the total number of correct top-1 predictions across all batches, computed under `no_grad`. Complete the line that adds this batch's correct count to the accumulator.

**Fill in:** adding the count of (preds == yb) matches to correct

In [ ]:
import torch as t
import torch.nn as nn

def count_correct(model, batches):
    correct = 0
    with t.no_grad():
        for xb, yb in batches:
            preds = model(xb).argmax(dim=1)
            correct += None  # TODO: adding the count of (preds == yb) matches to correct
    return correct

t.manual_seed(0)
m = nn.Linear(5, 3)
bs = [(t.randn(4, 5), t.randint(0, 3, (4,)))]
print(count_correct(m, bs))


def _test():
    t.manual_seed(0)
    m = nn.Linear(5, 3)
    batches = [(t.randn(4, 5), t.randint(0, 3, (4,))) for _ in range(3)]
    got = count_correct(m, batches)
    # independent ground truth recomputed directly
    expected = 0
    with t.no_grad():
        for xb, yb in batches:
            expected += int((m(xb).argmax(dim=1) == yb).sum())
    assert got == expected, (got, expected)
    # plausible bounds
    total = sum(yb.numel() for _, yb in batches)
    assert 0 <= got <= total


try:
    _test()
    _dd_passed.add('faded2')
    print('[Delta Drills] faded2 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
import torch as t
import torch.nn as nn

def count_correct(model, batches):
    correct = 0
    with t.no_grad():
        for xb, yb in batches:
            preds = model(xb).argmax(dim=1)
            correct += int((preds == yb).sum())
    return correct

t.manual_seed(0)
m = nn.Linear(5, 3)
bs = [(t.randn(4, 5), t.randint(0, 3, (4,)))]
print(count_correct(m, bs))
```
</details>